## Get started
This tutorial will show you how to import a model trained using snnTorch into asynctorch. Make sure asynctorch, snnTorch and tonic have been installed! Tonic is not an automatic dependency of asynctorch because it is not required for all use cases. For CUDA support, make sure you have a working CUDA installation and the correct version of PyTorch installed.

### Import packages

In [ ]:
import torch
import tonic
import snntorch.functional as SF
import tonic.transforms as transforms
from snntorch import surrogate
from asynctorch.nn.architecture.semi_sparse_fully_linear_architecture import SemiSparseFullyLinearArchitecture
from asynctorch.nn.neuron.lif_state import LIFState
from asynctorch.simulator.spike_scheduler import RandomSpikeScheduler
from asynctorch.simulator.spike_selector import SpikeSelector
from asynctorch.simulator.async_simulator import AsyncSimulator
from torch.utils.data import DataLoader
from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Load data

In [ ]:
timestep_size = 10000
transform = transforms.ToFrame(sensor_size=tonic.datasets.NMNIST.sensor_size, time_window=timestep_size)
test_dataset = tonic.datasets.NMNIST(save_to="./data", transform=transform, train=False, first_saccade_only=True)
collate_fn = tonic.collation.PadTensors(batch_first=False)
test_dataloader = DataLoader(test_dataset, batch_size=512, shuffle=True, collate_fn=collate_fn)

### Define the network architecture

In [ ]:
# Network architecture
n_inputs = 34 * 34 * 2
n_outputs = 10
neurons_per_layer = [64, 64, 64, n_outputs]
n_neurons = sum(neurons_per_layer)
network_module = SemiSparseFullyLinearArchitecture(n_inputs, neurons_per_layer, device)

# Neuron model
spike_grad = surrogate.straight_through_estimator()
state_module = LIFState(
    neurons_per_layer,
    tau_m=1000,
    membrane_threshold=0.3,
    sync_threshold=0.0,
    spike_grad=spike_grad,
    sync_grad=spike_grad,
    device=device,
)

# Simulator
spike_scheduler = RandomSpikeScheduler(state_module)
spike_selector_module = SpikeSelector(
    network_module,
    spike_scheduler,
    forward_group_size=16,
    device=device,
    prioritize_input=True,
)

## Start inferring
First, the function used for computing the accuracy of the model is defined. This function will be used to evaluate the model on the test set.

In [ ]:
accuracy_function = SF.accuracy_rate
def infer_and_print_accuracy(async_simulator: AsyncSimulator):
    async_simulator.eval()
    with torch.no_grad():
        accuracies = []
        batch_sizes = []
        for data, targets in tqdm(test_dataloader):
            data = data.to(device)
            targets = targets.to(device)
            async_simulator.reset_state()

            ys = []
            for t in range(data.shape[0]):
                ts_data = data[t].view(data[t].shape[0], -1)
                spk_out = async_simulator(ts_data, dt=timestep_size)[:, n_neurons-n_outputs:]
                ys.append(spk_out) 
            y = torch.stack(ys)
            accuracy = accuracy_function(y, targets)
            accuracies.append(accuracy)
            batch_sizes.append(data.shape[0])
        accuracy = sum([a*b for a, b in zip(accuracies, batch_sizes)]) / sum(batch_sizes)
        print(f"Accuracy: {accuracy}")

### Load and test a trained model from snnTorch
For snnTorch, the model is saved per layer. We will load the model layer by layer and convert it to asynctorch. This only works for fully-connected architectures.

In [ ]:
async_simulator = AsyncSimulator(state_module, spike_selector_module)
state_dict = torch.load("./nmnist_snntorch.pt")
weights_per_layer = []
sync_threshold = None
for key in state_dict:
    key_split = key.split(".")
    if key_split[-1] == "weight":
        weights_per_layer.append(state_dict[key].to(device))
architecture_state_dict = SemiSparseFullyLinearArchitecture.create_state_dict_from_weights_list(weights_per_layer)
state_dict = {
    "spike_selector.architecture.weights": architecture_state_dict["weights"],
    "spike_selector.architecture.input_weights": architecture_state_dict["input_weights"],
    "spike_selector.architecture.weights_mask": architecture_state_dict["weights_mask"],
}
async_simulator.load_state_dict(state_dict, strict=False)
infer_and_print_accuracy(async_simulator)

### Load and test a trained model from asynctorch
For asynctorch, the model can directly be loaded.

In [ ]:
async_simulator = AsyncSimulator(state_module, spike_selector_module)
state_dict = torch.load("./nmnist_asynctorch.pt")
async_simulator.load_state_dict(state_dict)
infer_and_print_accuracy(async_simulator)